# Bayesian Inital design

In [1]:
import numpy as np 
import matplotlib.pyplot as plt
from bayes_opt import BayesianOptimization

import tidy3d as td
import tidy3d.web as web

from main import (make_sim, get_coupling_efficiency, apodized_to_widths,
                    R, r0, initial_fill_factor, grating_period, 
                    etch_depth, to_substrate, n_wl, wl_range)

In [ ]:
def evaluate(R, r0, initial_fill_factor, etch_depth):
    """
    This function evaluates the FOM of the given parameters.
    """

    widths = apodized_to_widths(R=R, initial_fill_factor=initial_fill_factor)
    sim = make_sim(widths, r0=r0, etch_depth=etch_depth)
    sim_data = web.run(sim, task_name="bayesianOpt", verbose=False)
    
    return get_coupling_efficiency(sim_data)[n_wl//2]

    

## Minimum size of 50 nm

In [16]:
seed = 1234

init_points = 10
n_iter = 25

pbounds = {
    "R": (0.02, 0.08),
    "r0": (1.15, 1.25),
    "initial_fill_factor": (50/650, 0.3),
    "etch_depth": (0.05, 0.2)
}

defaults = {
    "R": R,
    "r0": 1.2,
    "initial_fill_factor": initial_fill_factor,
    "etch_depth": etch_depth
}

In [17]:
optimizer = BayesianOptimization(
    f=evaluate,
    pbounds=pbounds,
    random_state=seed,
    verbose=2
)

optimizer.probe(defaults, lazy=True)

In [18]:
optimizer.maximize(n_iter=n_iter, init_points=init_points)

|   iter    |  target   |     R     |    r0     | initia... | etch_d... |
-------------------------------------------------------------------------
| 1         | 0.4950663 | 0.03      | 1.2       | 0.1       | 0.1       |
| 2         | 0.0071463 | 0.0314911 | 1.2122108 | 0.1745700 | 0.1678037 |
| 3         | 0.0087336 | 0.0667985 | 1.1772592 | 0.1385958 | 0.1702808 |
| 4         | 0.0642456 | 0.0774883 | 1.2375932 | 0.1567438 | 0.1251492 |
| 5         | 0.0441306 | 0.0610077 | 1.2212702 | 0.1595174 | 0.1341794 |
| 6         | 0.0025610 | 0.0501849 | 1.1513768 | 0.2493228 | 0.1823961 |
| 7         | 0.4457126 | 0.0418931 | 1.2115396 | 0.0937388 | 0.1053236 |
| 8         | 0.0073070 | 0.0759884 | 1.2151378 | 0.1655298 | 0.1683095 |
| 9         | 0.0439796 | 0.0390101 | 1.2068098 | 0.2708053 | 0.1154260 |
| 10        | 0.0100642 | 0.0681288 | 1.1643766 | 0.2340274 | 0.1556871 |
| 11        | 0.0028527 | 0.0331275 | 1.2424867 | 0.1755544 | 0.1863973 |
| 12        | 0.0035194 | 0.0519986 | 

ValidationError: 3 validation errors for Box
size -> 0
  ensure this value is greater than or equal to 0 (type=value_error.number.not_ge; limit_value=0)
size -> 0
  instance of Box expected (type=type_error.arbitrary_type; expected_arbitrary_type=Box)
size
  instance of Box expected (type=type_error.arbitrary_type; expected_arbitrary_type=Box)

### for 50nm minimum, the original parameters are a good starting point

# minimum size of 100 nm

In [22]:
seed = 1234

init_points = 10
n_iter = 30

pbounds = {
    "R": (0.02, 0.065),
    "r0": (1.15, 1.25),
    "initial_fill_factor": (100/650, 0.25),
    "etch_depth": (0.05, 0.2)
}

defaults = {
    "R": R,
    "r0": 1.2,
    "initial_fill_factor": (100/650)*1.1,
    "etch_depth": etch_depth
}

In [23]:
optimizer = BayesianOptimization(
    f=evaluate,
    pbounds=pbounds,
    random_state=seed,
    verbose=2
)

optimizer.probe(defaults, lazy=True)

In [24]:
optimizer.maximize(n_iter=n_iter, init_points=init_points)

|   iter    |  target   |     R     |    r0     | initia... | etch_d... |
-------------------------------------------------------------------------
| 1         | 0.2980507 | 0.03      | 1.2       | 0.1692307 | 0.1       |
| 2         | 0.0059770 | 0.0286183 | 1.2122108 | 0.1959353 | 0.1678037 |
| 3         | 0.0059520 | 0.0550989 | 1.1772592 | 0.1804292 | 0.1702808 |
| 4         | 0.0498632 | 0.0631162 | 1.2375932 | 0.1882516 | 0.1251492 |
| 5         | 0.0323941 | 0.0507558 | 1.2212702 | 0.1894471 | 0.1341794 |
| 6         | 0.0027728 | 0.0426387 | 1.1513768 | 0.2281564 | 0.1823961 |
| 7         | 0.2479660 | 0.0364198 | 1.2115396 | 0.1610943 | 0.1053236 |
| 8         | 0.0064981 | 0.0619913 | 1.2151378 | 0.1920387 | 0.1683095 |
| 9         | 0.0614864 | 0.0342576 | 1.2068098 | 0.2374160 | 0.1154260 |
| 10        | 0.0105858 | 0.0560966 | 1.1643766 | 0.2215635 | 0.1556871 |
| 11        | 0.0025910 | 0.0298456 | 1.2424867 | 0.1963596 | 0.1863973 |
| 12        | 0.0038563 | 0.0439989 | 

### For the minimum of 100 nm, a good starting point is 

 - R = 0.02
 - r0 = 1.25
 - initial_fill_factor = 100/650 (probably pick a little larger since this is the bound)
 - etch_depth = 0.087
